<style>
ul { padding-left: 2.5rem; }
table { margin-left: 2.5rem !important; margin-right: 0 !important; }
th, td { text-align: left !important; }
li table, li div.course-figure { margin-left: 0 !important; }
.course-figure { width: auto; }
.katex-display { text-align: left !important; margin: 1rem 0 1.25rem 2.5rem !important; }
.katex-display > .katex, .katex-display > .katex > .katex-html { text-align: left !important; }
</style>

## 생명과학 예제: 생장 조건

### 1 · 문제와 정식화(problem formulation)

- 조명과 양액(nutrient solution) 조건(condition)을 골라 식물이 가장 많이 자라게 한다.
- 같은 품종(variety)을 같은 기간 동안 재배한다고 가정한다.
- 아래 값은 가상 생장량(growth)이며 단위는 mm이다.

| 조명 / 양액 | 낮음 (0) | 중간 (1) | 높음 (2) |
|:---|:---|:---|:---|
| 낮음 (0) | 3 | 5 | 4 |
| 중간 (1) | 6 | 9 | 7 |
| 높음 (2) | 5 | 8 | 6 |

- 의사결정벡터(decision vector)는 $\boldsymbol{x}=(x_1,x_2)$이며, 두 성분(component)은 조명과 양액의 단계이다.
- 목적함수(objective function) $f(\boldsymbol{x})$는 선택한 행과 열의 생장량이다.
- 각 단계는 0, 1, 2 중 하나를 선택한다.

$$
\max_{\boldsymbol{x}} f(\boldsymbol{x})
\qquad\text{subject to}\quad x_1,x_2\in\{0,1,2\}
$$

- 그림의 `[1, 2]`는 조명 중간·양액 높음이며 생장량은 7 mm이다.

<div class="course-figure" style="text-align: left; margin: 0.65rem 0 1.5rem 2.5rem;">
<img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/3c17b9cda768a9d4f9aaca66bc0b7d01e882a5da/05-2_metaheuristic_optimization_genetic_algorithm/assets/biology_encoding.svg" alt="Medium light and high nutrient levels give 7 mm growth in the example." width="440" style="display: block; width: 27.5em; max-width: 100%; height: auto; margin: 0;">
</div>


<style>
ul { padding-left: 2.5rem; }
table { margin-left: 2.5rem !important; margin-right: 0 !important; }
th, td { text-align: left !important; }
li table, li div.course-figure { margin-left: 0 !important; }
.course-figure { width: auto; }
.katex-display { text-align: left !important; margin: 1rem 0 1.25rem 2.5rem !important; }
.katex-display > .katex, .katex-display > .katex > .katex-html { text-align: left !important; }
</style>

### 2 · 코드 실행

- 공통 알고리듬(algorithm)을 정의한 뒤, 아래 셀에서 데이터(data)와 평가 함수(evaluation function)를 넣어 실행한다.


In [ ]:
import random

def genetic_algorithm(initial_population, gene_values, score, generations=20, seed=0):
    tournament_size = 2
    crossover_rate = 1.0
    mutation_rate = 0.3
    elite_count = 1
    blend_weight = 0.5
    mutation_scale = 0.1
    bounds = (-2.0, 2.0)

    # gene_values: allowed discrete values; None: real-valued genes.
    rng = random.Random(seed)
    population = initial_population.copy()
    population_size = len(population)

    for generation in range(generations):
        # Elitism: copy the best candidates without changing them.
        next_population = []
        remaining = population.copy()
        for elite in range(elite_count):
            best = remaining[0]
            for candidate in remaining:
                if score(candidate) > score(best):
                    best = candidate
            next_population.append(best.copy())
            remaining.remove(best)

        while len(next_population) < population_size:
            # Selection: sample candidates and choose the best in each sample.
            sample_a = rng.sample(population, tournament_size)
            sample_b = rng.sample(population, tournament_size)
            parent_a = sample_a[0]
            for candidate in sample_a:
                if score(candidate) > score(parent_a):
                    parent_a = candidate
            parent_b = sample_b[0]
            for candidate in sample_b:
                if score(candidate) > score(parent_b):
                    parent_b = candidate
            child = parent_a.copy()

            # Crossover: weighted average for real genes; join parts otherwise.
            if rng.random() < crossover_rate:
                if gene_values is None:
                    for i in range(len(child)):
                        child[i] = (blend_weight * parent_a[i]
                                    + (1 - blend_weight) * parent_b[i])
                else:
                    cut = rng.randrange(1, len(child))
                    for i in range(cut, len(child)):
                        child[i] = parent_b[i]

            # Mutation: change one randomly selected gene.
            if rng.random() < mutation_rate:
                i = rng.randrange(len(child))
                if gene_values is None:
                    child[i] += rng.uniform(-mutation_scale, mutation_scale)
                    if child[i] < bounds[0]:
                        child[i] = bounds[0]
                    if child[i] > bounds[1]:
                        child[i] = bounds[1]
                else:
                    alternatives = []
                    for value in gene_values:
                        if value != child[i]:
                            alternatives.append(value)
                    child[i] = rng.choice(alternatives)

            # Additional constraints: replace an infeasible child with its parent.
            if score(child) == float("-inf"):
                child = parent_a.copy()
            next_population.append(child)

        population = next_population

    best = population[0]
    for candidate in population:
        if score(candidate) > score(best):
            best = candidate
    return best.copy()


In [ ]:
growth = [[3, 5, 4], [6, 9, 7], [5, 8, 6]]

def evaluate(decision):
    light = decision[0]
    nutrient = decision[1]
    return {"light": light, "nutrient": nutrient,
            "growth": growth[light][nutrient]}

def score(decision):
    return evaluate(decision)["growth"]

initial_population = [[0, 0], [0, 2], [2, 0], [2, 2]]
best = genetic_algorithm(initial_population, [0, 1, 2], score)
result = evaluate(best)
print("Decision:", best)
print("Growth (mm):", result["growth"])
print("Stop reason: generation limit")


<style>
ul { padding-left: 2.5rem; }
table { margin-left: 2.5rem !important; margin-right: 0 !important; }
th, td { text-align: left !important; }
li table, li div.course-figure { margin-left: 0 !important; }
.course-figure { width: auto; }
.katex-display { text-align: left !important; margin: 1rem 0 1.25rem 2.5rem !important; }
.katex-display > .katex, .katex-display > .katex > .katex-html { text-align: left !important; }
</style>

- 결과는 `[1, 1]`이며 생장량은 9 mm이다.
- 이는 표의 아홉 조합(combination) 중 최대 생장량이다.
- 실제 식물이나 표 밖의 조건에 대한 최적성(optimality)은 보장하지 않는다.
